In [1]:
import yfinance as yf

data = yf.Ticker("AAPL").history(start="1996-01-01", end="2026-10-27")
data.index = data.index.tz_localize(None)
data = data.reset_index()
data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,1996-01-02,0.241146,0.241146,0.237408,0.240211,139294400,0.0,0.0
1,1996-01-03,0.239277,0.245820,0.238342,0.240211,429833600,0.0,0.0
2,1996-01-04,0.242082,0.242082,0.234604,0.236006,300182400,0.0,0.0
3,1996-01-05,0.236473,0.256102,0.234604,0.256102,445928000,0.0,0.0
4,1996-01-08,0.257971,0.265448,0.254232,0.258905,121340800,0.0,0.0


In [2]:
import numpy as np

data["Simple_Return"] = data["Close"].pct_change()
data["Log_Return"] = np.log(data["Close"] / data["Close"].shift(1))
data[["Date", "Close", "Simple_Return", "Log_Return"]].head()

,Date,Close,Simple_Return,Log_Return
0,1996-01-02,0.240211,NaN,NaN
1,1996-01-03,0.240211,0.000000,0.000000
2,1996-01-04,0.236006,-0.017508,-0.017663
3,1996-01-05,0.256102,0.085150,0.081718
4,1996-01-08,0.258905,0.010948,0.010889


In [4]:
import pandas as pd
import numpy as np
import yfinance as yf

# 1. AAPL (yfinance)
data = yf.Ticker("AAPL").history(start="1996-01-01", end="2026-10-27")
data.index = data.index.tz_localize(None)
data = data.reset_index()

# 2. CPI fresh load (FRED)
df_cpi = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL")
df_cpi.columns = ["date", "cpi"]
df_cpi["date"] = pd.to_datetime(df_cpi["date"])
df_cpi = df_cpi.set_index("date")

# 3. AAPL monthly last Close
data["Date"] = pd.to_datetime(data["Date"])
monthly = data.set_index("Date")["Close"].resample("MS").last().to_frame("Close")

# 4. Merge
merged = monthly.join(df_cpi, how="inner")

# 5. Returns
merged["simple_return"] = merged["Close"].pct_change()
merged["log_return"] = np.log(merged["Close"] / merged["Close"].shift(1))

# 6. Inflation (month-on-month)
merged["inflation"] = merged["cpi"].pct_change()

# 7. Real return
merged["real_return"] = (1 + merged["simple_return"]) / (1 + merged["inflation"]) - 1

merged = merged.reset_index().rename(columns={"index": "Date"})
merged.head()

,Date,Close,cpi,simple_return,log_return,inflation,real_return
0,1996-01-01,0.206564,154.7,NaN,NaN,NaN,NaN
1,1996-02-01,0.205629,155.0,-0.004525,-0.004535,0.001939,-0.006451
2,1996-03-01,0.183664,155.5,-0.106819,-0.112966,0.003226,-0.109691
3,1996-04-01,0.182262,156.1,-0.007633,-0.007663,0.003859,-0.011448
4,1996-05-01,0.195347,156.4,0.071795,0.069335,0.001922,0.069739
